# Estud-IA · IA nivel intermedio · Clase 3
**Cómo se representa un texto y cómo elegir un modelo**

Ejecuta las celdas en orden (Ctrl + Enter). Cinco partes:
1. Del texto a números: bolsa de palabras y n-gramas
2. Preparar el texto con spaCy: tokens, lemas y palabras vacías
3. Vectores: qué mide la similitud y qué no
4. Un modelo preentrenado con Hugging Face: leer su ficha antes de interpretar
5. Si trabajas con imágenes: el mismo criterio con una matriz de píxeles

Hilo de la clase: en la clase 2 la red falló con «no quiero devolver nada, solo saber dónde está mi pedido». Hoy buscamos por qué.

## Parte 1 · Del texto a números
Un modelo no lee palabras: recibe números. La **bolsa de palabras** cuenta qué palabras aparecen y **pierde el orden**.

**Contexto de lo que usas aquí:** `CountVectorizer` (scikit-learn) no es un modelo preentrenado: no aprende nada de otros textos, solo cuenta las palabras de las frases que le das. Funciona con cualquier idioma.

Antes de ejecutar: ¿en qué se diferencian los vectores de estas dos frases?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

frases = ["Quiero devolver el pedido", "No quiero devolver el pedido"]

bolsa = CountVectorizer()
X = bolsa.fit_transform(frases)
print("Vocabulario:", list(bolsa.get_feature_names_out()))
print(X.toarray())

Con **bigramas** (pares de palabras seguidas) se conserva un poco de orden: aparece «no quiero».

In [ ]:
bigramas = CountVectorizer(ngram_range=(1, 2))
X2 = bigramas.fit_transform(frases)
print("Columnas con bigramas:", X2.shape[1], "(antes:", X.shape[1], ")")
print([p for p in bigramas.get_feature_names_out() if "no" in p.split()])

## Parte 2 · Preparar el texto con spaCy
**Contexto del modelo `es_core_news_md` (versión 3.8.0):**

| Pregunta | Respuesta |
|---|---|
| ¿Qué es? | Un pipeline de spaCy para español, pensado para correr en CPU |
| ¿Idioma? | Español |
| ¿Tarea? | Tokens, lemas, categorías gramaticales, dependencias y entidades; incluye vectores de palabras |
| ¿Qué devuelve? | Anotaciones y vectores, **no** las etiquetas de tu proyecto |
| ¿Con qué aprendió? | UD Spanish AnCora y WikiNER (textos anotados) y vectores fastText de 300 dimensiones |
| ¿Tamaño? | Unos 54 MB instalado |

Fuente: metadatos del modelo instalado (licencia GNU GPL 3.0).

spaCy es una biblioteca de **procesamiento**: separa el texto en unidades (tokens), calcula lemas y marca palabras muy frecuentes llamadas *palabras vacías*.

**Reto:** si quitamos las palabras vacías antes de representar el texto, ¿en qué quedan las dos frases? Predice antes de ejecutar.

In [ ]:
# En Colab, ejecuta esta línea una vez:
# !python -m spacy download es_core_news_md
import spacy

nlp = spacy.load("es_core_news_md")

for frase in frases:
    doc = nlp(frase)
    print(frase)
    for t in doc:
        print(f"  {t.text:10s} lema={t.lemma_:10s} categoría={t.pos_:6s} ¿palabra vacía?={t.is_stop}")

In [ ]:
for frase in frases:
    doc = nlp(frase)
    print(f"{frase!r:34s} -> sin palabras vacías:", [t.text.lower() for t in doc if not t.is_stop])

**Observa:** la preparación puede quitar justo la palabra que cambia la intención. Preparar es tomar una decisión, no solo limpiar.

## Parte 3 · Vectores y similitud
El modelo `es_core_news_md` (español, vectores de 300 dimensiones; ver su ficha en la Parte 2) trae un vector por palabra. La similitud de una frase es el **promedio** de sus vectores de palabras.

**Cercano no es igual:** una similitud alta no garantiza que el significado sea el mismo.

In [ ]:
pares = [
    ("Quiero devolver el pedido", "No quiero devolver el pedido"),
    ("Quiero devolver el pedido", "Deseo regresar la compra"),
    ("Quiero devolver el pedido", "La factura llegó con otro nombre"),
]
for a, b in pares:
    print(f"{nlp(a).similarity(nlp(b)):.2f}  {a!r} vs {b!r}")

Antes de seguir: la primera pareja significa lo contrario y la segunda significa lo mismo. ¿Qué dice la similitud de cada una? ¿Sirve para decidir la intención?

## Parte 4 · Un modelo preentrenado con Hugging Face
`transformers` carga modelos ya entrenados y su propio tokenizador, que parte las palabras en **subpalabras**. Antes de interpretar una salida, lee la **ficha del modelo** (huggingface.co) y responde:

| Pregunta | Dónde la encuentras |
|---|---|
| ¿Para qué idioma? | Ficha del modelo |
| ¿Para qué tarea? | Ficha del modelo |
| ¿Qué etiquetas devuelve? | Ficha o `config` del modelo |
| ¿Con qué datos aprendió? | Ficha del modelo |

**Contexto del modelo `nlptown/bert-base-multilingual-uncased-sentiment`:**

| Pregunta | Respuesta |
|---|---|
| ¿Qué es? | BERT multilingüe (no distingue mayúsculas) afinado para sentimiento |
| ¿Idioma? | Seis: inglés, neerlandés, alemán, francés, italiano y español |
| ¿Tarea? | Sentimiento de reseñas de producto |
| ¿Qué devuelve? | De 1 a 5 estrellas («1 star» … «5 stars») |
| ¿Con qué aprendió? | Unas 629 mil reseñas de producto; unas 50 mil en español |
| ¿Tamaño? | Unos 0,2 mil millones de parámetros (≈ 200 millones) |
| ¿Qué tan bien? | En español acierta las estrellas exactas en 58 % y con error de una estrella en 95 %, medido en reseñas |

Fuente: ficha del modelo en Hugging Face (licencia MIT). Esas cifras son sobre reseñas, no sobre tus mensajes.

Es un modelo de **sentimiento**, no de tus categorías (facturación, envío, devolución). Lee sus etiquetas antes de ejecutar.

In [ ]:
# En Colab, ya viene instalado. Si falta: !pip -q install transformers
from transformers import AutoTokenizer, pipeline

MODELO = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizador = AutoTokenizer.from_pretrained(MODELO)

for texto in ["No quiero devolver el pedido", "parce, nada que me llega el chunche"]:
    print(texto)
    print("  subpalabras:", tokenizador.tokenize(texto))

In [ ]:
clasificador = pipeline("text-classification", model=MODELO)

textos = [
    "Quiero devolver el pedido",
    "No quiero devolver el pedido",
    "El envío llegó rápido",
    "El envío no llegó rápido",
    "parce, nada que me llega el chunche",
]
for t in textos:
    r = clasificador(t)[0]
    print(f"{r['label']:8s} {r['score']:.0%}  {t}")

**Preguntas para escribir en el chat:**
1. ¿Las etiquetas del modelo (estrellas) son las de tu proyecto?
2. ¿Qué cambió entre una frase y su versión con «no»?
3. ¿Qué puedes afirmar y qué todavía no?

## Parte 5 · Si trabajas con imágenes
El mismo criterio: **preparar es decidir qué información se conserva.** Usamos el «2» de 5 × 5 de la clase 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dos = np.array([
    [255, 255, 255, 255, 255],
    [  0,   0,   0,   0, 255],
    [255, 255, 255, 255, 255],
    [255,   0,   0,   0,   0],
    [255, 255, 255, 255, 255],
])

versiones = {
    "Original": dos,
    "Normalizada (0 a 1)": dos / 255,
    "Volteada": np.fliplr(dos),
}
fig, ejes = plt.subplots(1, 3, figsize=(9, 3))
for eje, (nombre, img) in zip(ejes, versiones.items()):
    eje.imshow(img, cmap="gray", vmin=0, vmax=img.max())
    eje.set_title(nombre)
    eje.axis("off")
plt.show()

In [ ]:
# Bajar la resolución: promedio de bloques de 2 x 2 sobre una imagen de 4 x 4
chica = np.array([
    [255, 255,   0,   0],
    [255, 255,   0,   0],
    [  0,   0, 255, 255],
    [  0,   0, 255, 255],
])
reducida = chica.reshape(2, 2, 2, 2).mean(axis=(1, 3))
print(reducida)

**Observa:** normalizar cambia la escala pero no el contenido; voltear cambia el contenido; reducir la resolución puede mezclar detalles. ¿Cuál de estos cambios conservaría lo que tu modelo necesita?

## Tu turno
1. Escribe un caso difícil de **tu** proyecto (una negación, una jerga o una foto con sombra) y pásalo por la preparación que usarías.
2. Anota qué se pierde con esa preparación y qué conserva la representación.
3. Elige un modelo candidato y completa su ficha: idioma, tarea, etiquetas y datos de entrenamiento.
4. Escribe una conclusión: **la preparación y la representación deciden qué puede distinguir el modelo.**